# SigLIP2 + LoRA 이미지 순서 예측

이 파일은 `# %%` 단위로 나뉘어 있습니다. VS Code, Spyder 또는 함께
제공되는 `.ipynb`에서 1번부터 순서대로 한 셀씩 실행하세요.



## 1. 환경 설정 및 패키지 설치

인터넷이 되는 개발 환경에서 처음 한 번만 실행합니다. 평가 환경에서는
이미 준비한 프로젝트 폴더를 옮긴 뒤 두 옵션을 `False`로 설정합니다.



In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = (
    Path(__file__).resolve().parent
    if "__file__" in globals()
    else Path.cwd().resolve()
)
if not (PROJECT_DIR / "common.py").is_file():
    candidate = PROJECT_DIR / "outputs" / "siglip2_ordering"
    if candidate.is_dir():
        PROJECT_DIR = candidate.resolve()

INSTALL_PACKAGES = True
DOWNLOAD_MODEL = True

MODEL_DIR = PROJECT_DIR / "pretrained" / "siglip2-base-patch16-224"

if INSTALL_PACKAGES:
    # Notebook 하나만 복사해도 설치할 수 있도록 목록을 셀 안에 포함합니다.
    required_packages = [
        "torch==2.5.1",
        "torchvision==0.20.1",
        "torchaudio==2.5.1",
        "transformers==4.53.3",
        "peft==0.15.2",
        "accelerate==1.8.1",
        "pandas==2.2.3",
        "numpy==2.1.3",
        "scikit-learn==1.6.1",
        "Pillow==11.1.0",
        "tqdm==4.67.1",
        "safetensors==0.5.3",
        "sentencepiece==0.2.0",
        "huggingface-hub==0.33.2",
        "ipykernel==6.29.5",
    ]
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            *required_packages,
        ]
    )

if DOWNLOAD_MODEL and not (MODEL_DIR / "model.safetensors").is_file():
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id="google/siglip2-base-patch16-224",
        revision="75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2",
        local_dir=MODEL_DIR,
        allow_patterns=[
            "*.json",
            "*.model",
            "*.safetensors",
            "README.md",
            "LICENSE*",
        ],
    )
    weight_path = MODEL_DIR / "model.safetensors"
    if not weight_path.is_file() or weight_path.stat().st_size < 1_000_000_000:
        raise RuntimeError("SigLIP2 가중치 다운로드가 완전하지 않습니다.")

print("Project directory:", PROJECT_DIR)
print("Python:", sys.executable)
print("Model directory:", MODEL_DIR)



## 2. 라이브러리 임포트 및 경로 설정

`SNU_DATA_DIR` 환경변수가 있으면 그 경로를 사용하고, 없으면 저장소의
`data/` 폴더를 사용합니다. 데이터와 제출 파일은 Git에 포함하지 않습니다.



In [ ]:
import ast
import itertools
import json
import math
import os
import random
import time
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoProcessor,
    get_cosine_schedule_with_warmup,
)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

DATA_DIR = Path(
    os.environ.get(
        "SNU_DATA_DIR",
        PROJECT_DIR / "data",
    )
)
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMAGE_DIR = DATA_DIR / "train"
TEST_CSV = DATA_DIR / "test.csv"
TEST_IMAGE_DIR = DATA_DIR / "test"
SAMPLE_SUBMISSION_CSV = DATA_DIR / "sample_submission.csv"
OUTPUT_DIR = PROJECT_DIR / "artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)



## 3. 재현성 설정 및 라벨 정의



In [ ]:
from __future__ import annotations

import ast
import itertools
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageEnhance
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset


ORDERS = list(itertools.permutations(range(4)))
ORDER_TO_CLASS = {order: idx for idx, order in enumerate(ORDERS)}
CLASS_TO_ORDER = {
    idx: [item + 1 for item in order] for idx, order in enumerate(ORDERS)
}
PERMUTATIONS = torch.tensor(ORDERS, dtype=torch.long)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def parse_answer(answer) -> tuple[int, int, int, int]:
    if isinstance(answer, str):
        answer = ast.literal_eval(answer)
    order = tuple(int(item) - 1 for item in answer)
    if sorted(order) != [0, 1, 2, 3]:
        raise ValueError(f"Invalid answer: {answer}")
    return order


def parse_answer_to_class(answer) -> int:
    return ORDER_TO_CLASS[parse_answer(answer)]


def class_to_answer(pred_class: int) -> list[int]:
    return CLASS_TO_ORDER[int(pred_class)]


class ConsistentWeakAugmentation:
    """Apply one weak photometric transform consistently to all four images."""

    def sample(self):
        if random.random() >= 0.70:
            return None
        return {
            "brightness": random.uniform(0.90, 1.10),
            "contrast": random.uniform(0.90, 1.10),
            "color": random.uniform(0.95, 1.05),
        }

    @staticmethod
    def apply(image: Image.Image, params) -> Image.Image:
        if params is None:
            return image
        image = ImageEnhance.Brightness(image).enhance(params["brightness"])
        image = ImageEnhance.Contrast(image).enhance(params["contrast"])
        return ImageEnhance.Color(image).enhance(params["color"])


class OrderingDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        image_dir: Path,
        with_labels: bool,
        is_train: bool,
    ):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.with_labels = with_labels
        self.augmentation = ConsistentWeakAugmentation() if is_train else None

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        params = self.augmentation.sample() if self.augmentation else None
        images = []

        for column in ("Input_1", "Input_2", "Input_3", "Input_4"):
            image_path = self.image_dir / str(row["Id"]) / str(row[column])
            if not image_path.is_file():
                raise FileNotFoundError(f"Image not found: {image_path}")
            with Image.open(image_path) as image:
                image = image.convert("RGB")
                if self.augmentation:
                    image = self.augmentation.apply(image, params)
                images.append(image.copy())

        item = {
            "id": row["Id"],
            "sentence": str(row["Sentence"]),
            "images": images,
        }
        if self.with_labels:
            item["labels"] = parse_answer_to_class(row["Answer"])
        return item


class Siglip2OrderingCollator:
    def __init__(self, processor, max_length: int = 64):
        self.processor = processor
        self.max_length = max_length

    def __call__(self, batch: list[dict]) -> dict:
        # SigLIP2 was trained with lowercasing and max-length padding.
        texts = [item["sentence"].lower() for item in batch]
        images = [image for item in batch for image in item["images"]]

        text_inputs = self.processor(
            text=texts,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        image_inputs = self.processor(images=images, return_tensors="pt")

        batch_size = len(batch)
        pixel_values = image_inputs["pixel_values"].reshape(
            batch_size,
            4,
            *image_inputs["pixel_values"].shape[1:],
        )
        input_ids = text_inputs["input_ids"]
        # The pinned SigLIP2 tokenizer intentionally lists only input_ids in
        # model_input_names and may not emit an attention mask. All-ones is
        # equivalent to the model's native attention_mask=None behavior.
        attention_mask = text_inputs.get(
            "attention_mask",
            torch.ones_like(input_ids),
        )
        result = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "pixel_values": pixel_values,
            "ids": [item["id"] for item in batch],
        }
        if "labels" in batch[0]:
            result["labels"] = torch.tensor(
                [item["labels"] for item in batch],
                dtype=torch.long,
            )
        return result


class Siglip2LoRAPermutationRanker(nn.Module):
    """One SigLIP2 backbone with LoRA and one permutation-ranking head."""

    def __init__(
        self,
        siglip_model,
        lora_r: int = 16,
        lora_alpha: int = 32,
        lora_dropout: float = 0.05,
        head_dropout: float = 0.15,
        pair_loss_weight: float = 0.35,
        label_smoothing: float = 0.02,
    ):
        super().__init__()
        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias="none",
            target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
        )
        self.siglip = get_peft_model(siglip_model, lora_config)
        self.pair_loss_weight = pair_loss_weight
        self.label_smoothing = label_smoothing

        text_config = siglip_model.config.text_config
        hidden = getattr(
            text_config,
            "projection_size",
            text_config.hidden_size,
        )

        sequence_dim = hidden * 11 + 4
        self.sequence_scorer = nn.Sequential(
            nn.LayerNorm(sequence_dim),
            nn.Linear(sequence_dim, 1024),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(1024, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )

        pair_dim = hidden * 5 + 2
        self.pair_scorer = nn.Sequential(
            nn.LayerNorm(pair_dim),
            nn.Linear(pair_dim, 512),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

        # The data contains a strong [1,2,3,4] prior. This learnable bias lets
        # the single model use training-set priors without looking at test data.
        self.class_bias = nn.Parameter(torch.zeros(24))
        self.log_pair_weight = nn.Parameter(torch.zeros(()))
        self.register_buffer("permutations", PERMUTATIONS.clone(), persistent=False)

        pair_masks = []
        for permutation in ORDERS:
            mask = torch.zeros(4, 4, dtype=torch.float32)
            position = {item: pos for pos, item in enumerate(permutation)}
            for i in range(4):
                for j in range(4):
                    if i != j and position[i] < position[j]:
                        mask[i, j] = 1.0
            pair_masks.append(mask)
        self.register_buffer(
            "permutation_pair_masks",
            torch.stack(pair_masks),
            persistent=False,
        )

    def core(self):
        return self.siglip.base_model.model

    def initialize_class_prior(self, class_counts: np.ndarray) -> None:
        probabilities = (class_counts + 1.0) / (class_counts.sum() + 24.0)
        prior = torch.tensor(np.log(probabilities), dtype=self.class_bias.dtype)
        prior -= prior.mean()
        with torch.no_grad():
            self.class_bias.copy_(prior)

    def encode(self, input_ids, attention_mask, pixel_values):
        batch_size = input_ids.shape[0]
        core = self.core()
        text_outputs = core.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        image_outputs = core.vision_model(
            pixel_values=pixel_values.flatten(0, 1),
        )
        text_embeddings = F.normalize(
            text_outputs.pooler_output.float(),
            dim=-1,
        )
        image_embeddings = F.normalize(
            image_outputs.pooler_output.float().reshape(batch_size, 4, -1),
            dim=-1,
        )
        return text_embeddings, image_embeddings

    def compute_pair_logits(self, text_embeddings, image_embeddings, similarities):
        image_i = image_embeddings[:, :, None, :].expand(-1, 4, 4, -1)
        image_j = image_embeddings[:, None, :, :].expand(-1, 4, 4, -1)
        text = text_embeddings[:, None, None, :].expand(-1, 4, 4, -1)
        similarity_i = similarities[:, :, None, None].expand(-1, 4, 4, -1)
        similarity_j = similarities[:, None, :, None].expand(-1, 4, 4, -1)

        features = torch.cat(
            [
                text,
                image_i,
                image_j,
                image_j - image_i,
                image_i * image_j,
                similarity_i,
                similarity_j,
            ],
            dim=-1,
        )
        raw_logits = self.pair_scorer(features).squeeze(-1)
        # Enforce P(i before j) = 1 - P(j before i).
        pair_logits = 0.5 * (raw_logits - raw_logits.transpose(1, 2))
        eye = torch.eye(4, device=pair_logits.device, dtype=torch.bool)
        return pair_logits.masked_fill(eye.unsqueeze(0), 0.0)

    def compute_permutation_logits(
        self,
        text_embeddings,
        image_embeddings,
        similarities,
        pair_logits,
    ):
        batch_size = image_embeddings.shape[0]
        permutation_index = self.permutations[None].expand(batch_size, -1, -1)
        batch_index = torch.arange(
            batch_size,
            device=image_embeddings.device,
        )[:, None, None]
        ordered_images = image_embeddings[batch_index, permutation_index]
        ordered_similarities = similarities[batch_index, permutation_index]

        adjacent_differences = ordered_images[:, :, 1:] - ordered_images[:, :, :-1]
        first_last_difference = ordered_images[:, :, -1] - ordered_images[:, :, 0]
        text_expanded = text_embeddings[:, None, :].expand(-1, 24, -1)
        text_order_interaction = (
            ordered_images * text_embeddings[:, None, None, :]
        ).mean(dim=2)
        image_std = image_embeddings.std(dim=1)[:, None, :].expand(-1, 24, -1)

        sequence_features = torch.cat(
            [
                text_expanded,
                ordered_images.flatten(2),
                adjacent_differences.flatten(2),
                first_last_difference,
                ordered_similarities,
                text_order_interaction,
                image_std,
            ],
            dim=-1,
        )
        sequence_logits = self.sequence_scorer(sequence_features).squeeze(-1)
        masks = self.permutation_pair_masks.to(dtype=pair_logits.dtype)
        pair_scores = torch.einsum("bij,pij->bp", pair_logits, masks) / 6.0
        pair_weight = self.log_pair_weight.exp()
        return sequence_logits + pair_weight * pair_scores + self.class_bias

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        pixel_values=None,
        labels=None,
    ):
        text_embeddings, image_embeddings = self.encode(
            input_ids,
            attention_mask,
            pixel_values,
        )
        similarities = torch.einsum(
            "bd,bnd->bn",
            text_embeddings,
            image_embeddings,
        )
        pair_logits = self.compute_pair_logits(
            text_embeddings,
            image_embeddings,
            similarities,
        )
        logits = self.compute_permutation_logits(
            text_embeddings,
            image_embeddings,
            similarities,
            pair_logits,
        )
        output = {"logits": logits, "pair_logits": pair_logits}

        if labels is not None:
            class_loss = F.cross_entropy(
                logits,
                labels,
                label_smoothing=self.label_smoothing,
            )
            targets = self.permutation_pair_masks[labels]
            non_diagonal = ~torch.eye(
                4,
                device=labels.device,
                dtype=torch.bool,
            ).unsqueeze(0).expand(labels.shape[0], -1, -1)
            pair_loss = F.binary_cross_entropy_with_logits(
                pair_logits[non_diagonal],
                targets[non_diagonal].to(pair_logits.dtype),
            )
            output["loss"] = class_loss + self.pair_loss_weight * pair_loss
        return output


def trainable_state_dict(model: nn.Module) -> dict[str, torch.Tensor]:
    trainable_names = {
        name for name, parameter in model.named_parameters() if parameter.requires_grad
    }
    return {
        name: value.detach().cpu()
        for name, value in model.state_dict().items()
        if name in trainable_names
    }


def move_batch_to_device(batch: dict, device: torch.device) -> dict:
    return {
        key: value.to(device, non_blocking=True)
        for key, value in batch.items()
        if key != "ids"
    }


SEED = 42
VALID_SIZE = 0.15
BATCH_SIZE = 4
GRAD_ACCUM = 8
EPOCHS = 12
PATIENCE = 4
HEAD_LR = 3e-4
LORA_LR = 8e-5
WEIGHT_DECAY = 0.02
WARMUP_RATIO = 0.08
LORA_R = 16
LORA_ALPHA = 32
NUM_WORKERS = 0 if sys.platform == "darwin" else min(8, os.cpu_count() or 0)

seed_everything(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else "cpu"
)
USE_CUDA = DEVICE.type == "cuda"
AMP_DTYPE = (
    torch.bfloat16
    if USE_CUDA and torch.cuda.is_bf16_supported()
    else torch.float16
)
if USE_CUDA:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = False

ORDER_TO_CLASS = {order: idx for idx, order in enumerate(ORDERS)}
print("Device:", DEVICE)
print("Number of permutation labels:", len(ORDERS))
print("Example:", ORDERS[0], "-> class 0 ->", CLASS_TO_ORDER[0])



## 4. 학습 데이터 불러오기 및 파일 검사



In [ ]:
if not TRAIN_CSV.is_file():
    raise FileNotFoundError(f"train.csv가 없습니다: {TRAIN_CSV}")
if not TRAIN_IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"train 이미지 폴더가 없습니다: {TRAIN_IMAGE_DIR}")

train_df = pd.read_csv(TRAIN_CSV)
required_columns = {
    "Id",
    "Input_1",
    "Input_2",
    "Input_3",
    "Input_4",
    "Sentence",
    "Answer",
}
missing_columns = required_columns - set(train_df.columns)
if missing_columns:
    raise ValueError(f"train.csv에 필요한 열이 없습니다: {sorted(missing_columns)}")

train_df["_class"] = train_df["Answer"].map(parse_answer_to_class)

missing_images = []
for row in train_df.itertuples(index=False):
    for column in ("Input_1", "Input_2", "Input_3", "Input_4"):
        image_path = TRAIN_IMAGE_DIR / str(row.Id) / str(getattr(row, column))
        if not image_path.is_file():
            missing_images.append(image_path)
            if len(missing_images) >= 20:
                break
    if len(missing_images) >= 20:
        break
if missing_images:
    raise FileNotFoundError(
        "학습 이미지가 없습니다:\n" + "\n".join(map(str, missing_images))
    )

class_counts_all = train_df["_class"].value_counts().sort_index()
identity_count = int((train_df["_class"] == 0).sum())
print("Train rows:", len(train_df))
print("Classes:", train_df["_class"].nunique())
print(
    "[1,2,3,4] prior:",
    identity_count,
    f"({identity_count / len(train_df):.2%})",
)
display(train_df.head(3)) if "display" in globals() else print(train_df.head(3))



## 5. 학습/검증 데이터 분리



In [ ]:
train_part, valid_part = train_test_split(
    train_df,
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=train_df["_class"],
)
print("Train split:", len(train_part))
print("Valid split:", len(valid_part))



## 6. SigLIP2 프로세서, 데이터셋 및 DataLoader



In [ ]:
if not MODEL_DIR.is_dir():
    raise FileNotFoundError(
        f"모델 폴더가 없습니다: {MODEL_DIR}\n"
        "1번 셀에서 DOWNLOAD_MODEL=True로 실행하세요."
    )

processor = AutoProcessor.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    use_fast=False,
)
collator = Siglip2OrderingCollator(processor)

train_dataset = OrderingDataset(
    train_part,
    TRAIN_IMAGE_DIR,
    with_labels=True,
    is_train=True,
)
valid_dataset = OrderingDataset(
    valid_part,
    TRAIN_IMAGE_DIR,
    with_labels=True,
    is_train=False,
)

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": USE_CUDA,
    "persistent_workers": NUM_WORKERS > 0,
    "collate_fn": collator,
    "worker_init_fn": seed_worker,
}
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    **loader_kwargs,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    **loader_kwargs,
)

sample_batch = next(iter(train_loader))
print("input_ids:", tuple(sample_batch["input_ids"].shape))
print("pixel_values:", tuple(sample_batch["pixel_values"].shape))
print("labels:", tuple(sample_batch["labels"].shape))



## 7. 단일 SigLIP2 + LoRA 순열 모델 생성



In [ ]:
base_model = AutoModel.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    torch_dtype=torch.float32,
)

# OOM이 발생하면 다음 줄의 주석을 해제한 후 7번부터 다시 실행합니다.
# base_model.gradient_checkpointing_enable()

model = Siglip2LoRAPermutationRanker(
    base_model,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
)
train_class_counts = np.bincount(train_part["_class"], minlength=24)
model.initialize_class_prior(train_class_counts)
model.to(DEVICE)
model.siglip.print_trainable_parameters()

with torch.inference_mode():
    smoke_batch = {
        key: value[:1].to(DEVICE)
        for key, value in sample_batch.items()
        if key != "ids"
    }
    smoke_output = model(**smoke_batch)
print("Permutation logits:", tuple(smoke_output["logits"].shape))
print("Initial loss:", smoke_output["loss"].item())



## 8. Optimizer, Scheduler 및 AMP 설정



In [ ]:
lora_parameters = []
head_parameters = []
for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    if "lora_" in name:
        lora_parameters.append(parameter)
    else:
        head_parameters.append(parameter)

optimizer = torch.optim.AdamW(
    [
        {"params": lora_parameters, "lr": LORA_LR},
        {"params": head_parameters, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.98),
)
optimizer_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = optimizer_steps_per_epoch * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps,
)
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_CUDA and AMP_DTYPE == torch.float16,
)


def autocast_context():
    if USE_CUDA:
        return torch.autocast("cuda", dtype=AMP_DTYPE)
    return nullcontext()


print("Effective batch size:", BATCH_SIZE * GRAD_ACCUM)
print("Optimizer steps per epoch:", optimizer_steps_per_epoch)
print("Total optimizer steps:", total_steps)



## 9. 검증 함수 정의



In [ ]:
@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    loss_sum = 0.0
    correct = 0
    count = 0

    for raw_batch in tqdm(loader, desc="Validation", leave=False):
        batch = move_batch_to_device(raw_batch, DEVICE)
        with autocast_context():
            output = model(**batch)
        batch_size = batch["labels"].shape[0]
        loss_sum += output["loss"].item() * batch_size
        correct += (
            output["logits"].argmax(-1) == batch["labels"]
        ).sum().item()
        count += batch_size

    return loss_sum / count, correct / count


initial_valid_loss, initial_valid_accuracy = evaluate(model, valid_loader)
print("Initial valid loss:", initial_valid_loss)
print("Initial valid accuracy:", initial_valid_accuracy)



## 10. 모델 학습 및 최적 체크포인트 저장



In [ ]:
BEST_MODEL_PATH = OUTPUT_DIR / "best_model.pt"
HISTORY_PATH = OUTPUT_DIR / "history.json"
best_valid_accuracy = -1.0
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_loss_sum = 0.0
    train_correct = 0
    train_count = 0

    remainder = len(train_loader) % GRAD_ACCUM
    remainder_start = len(train_loader) - remainder + 1 if remainder else None
    progress = tqdm(
        enumerate(train_loader, start=1),
        total=len(train_loader),
        desc=f"Epoch {epoch}/{EPOCHS}",
    )

    for step, raw_batch in progress:
        batch = move_batch_to_device(raw_batch, DEVICE)
        divisor = (
            remainder
            if remainder_start is not None and step >= remainder_start
            else GRAD_ACCUM
        )
        with autocast_context():
            output = model(**batch)
            scaled_loss = output["loss"] / divisor
        scaler.scale(scaled_loss).backward()

        batch_size = batch["labels"].shape[0]
        train_loss_sum += output["loss"].detach().item() * batch_size
        train_correct += (
            output["logits"].detach().argmax(-1) == batch["labels"]
        ).sum().item()
        train_count += batch_size

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                1.0,
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        progress.set_postfix(
            loss=f"{train_loss_sum / train_count:.4f}",
            accuracy=f"{train_correct / train_count:.4f}",
        )

    train_loss = train_loss_sum / train_count
    train_accuracy = train_correct / train_count
    valid_loss, valid_accuracy = evaluate(model, valid_loader)
    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_accuracy,
    }
    history.append(record)
    HISTORY_PATH.write_text(
        json.dumps(history, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(record)

    if valid_accuracy > best_valid_accuracy:
        best_valid_accuracy = valid_accuracy
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": trainable_state_dict(model),
                "metadata": {
                    "backbone": MODEL_DIR.name,
                    "lora_r": LORA_R,
                    "lora_alpha": LORA_ALPHA,
                    "best_epoch": epoch,
                    "best_valid_accuracy": best_valid_accuracy,
                },
            },
            BEST_MODEL_PATH,
        )
        print("Saved:", BEST_MODEL_PATH)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping")
            break

print("Best validation accuracy:", best_valid_accuracy)



## 11. 최적 체크포인트 불러오기



In [ ]:
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu",
    weights_only=True,
)
load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=False,
)
if load_result.unexpected_keys:
    raise RuntimeError(f"Unexpected checkpoint keys: {load_result.unexpected_keys}")
model.to(DEVICE).eval()
print("Loaded epoch:", checkpoint["metadata"]["best_epoch"])
print("Loaded valid accuracy:", checkpoint["metadata"]["best_valid_accuracy"])



## 12. 테스트 데이터 추론

공식 `test/<Id>/이미지` 폴더가 있어야 실행됩니다. 현재 로컬 폴더에 test
이미지가 없다면 대회에서 받은 원본 파일을 먼저 배치해야 합니다.



In [ ]:
if not TEST_CSV.is_file():
    raise FileNotFoundError(f"test.csv가 없습니다: {TEST_CSV}")
if not TEST_IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        f"공식 test 이미지 폴더가 없습니다: {TEST_IMAGE_DIR}"
    )

test_df = pd.read_csv(TEST_CSV)
test_dataset = OrderingDataset(
    test_df,
    TEST_IMAGE_DIR,
    with_labels=False,
    is_train=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    **loader_kwargs,
)

test_ids = []
test_answers = []
inference_started = time.perf_counter()
with torch.inference_mode():
    for raw_batch in tqdm(test_loader, desc="Inference"):
        test_ids.extend(raw_batch["ids"])
        batch = move_batch_to_device(raw_batch, DEVICE)
        with autocast_context():
            predictions = model(**batch)["logits"].argmax(-1).cpu().tolist()
        test_answers.extend(
            str(class_to_answer(prediction)) for prediction in predictions
        )

inference_hours = (time.perf_counter() - inference_started) / 3600
print("Predictions:", len(test_answers))
print(f"Inference time: {inference_hours:.3f} hours")



## 13. 제출 파일 생성 및 형식 검사



In [ ]:
answer_by_id = dict(zip(test_ids, test_answers))
if SAMPLE_SUBMISSION_CSV.is_file():
    submission = pd.read_csv(SAMPLE_SUBMISSION_CSV)
    if "Id" not in submission.columns or "Answer" not in submission.columns:
        raise ValueError("sample_submission.csv에는 Id와 Answer 열이 필요합니다.")
    missing_predictions = set(submission["Id"]) - set(answer_by_id)
    if missing_predictions:
        raise ValueError(
            f"예측이 없는 Id가 {len(missing_predictions)}개 있습니다."
        )
    submission["Answer"] = submission["Id"].map(answer_by_id)
else:
    submission = pd.DataFrame(
        {
            "Id": test_ids,
            "Answer": test_answers,
        }
    )

if len(submission) != len(test_df):
    raise ValueError("제출 행 수와 test.csv 행 수가 다릅니다.")
if submission["Answer"].isna().any():
    raise ValueError("제출 파일에 빈 Answer가 있습니다.")

SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved:", SUBMISSION_PATH)
display(submission.head()) if "display" in globals() else print(submission.head())
